In [1]:
import pandas as pd
import sqlite3
import pickle

conn = sqlite3.connect("../database/yieldwatch.db")

# Load the saved model artifacts from Phase 10
with open("../models/defect_model.pkl", "rb") as f:
    rf = pickle.load(f)
with open("../models/feature_columns.pkl", "rb") as f:
    feature_columns = pickle.load(f)

runs = pd.read_sql("SELECT * FROM production_runs", conn)

print("Data and model loaded.")
print("Model type:", type(rf).__name__)
print("Number of features:", len(feature_columns))

Data and model loaded.
Model type: RandomForestClassifier
Number of features: 27


In [2]:
plant_defect_rate = runs["defective_units"].sum() / runs["produced_units"].sum() * 100

worst_machine = runs.groupby("machine_id").apply(
    lambda g: g["defective_units"].sum() / g["produced_units"].sum() * 100, include_groups=False
).sort_values(ascending=False)

worst_shift = runs.groupby("shift_id").apply(
    lambda g: g["defective_units"].sum() / g["produced_units"].sum() * 100, include_groups=False
).sort_values(ascending=False)

worst_material = runs.groupby("material_id").apply(
    lambda g: g["defective_units"].sum() / g["produced_units"].sum() * 100, include_groups=False
).sort_values(ascending=False)

worst_combo = runs.groupby(["machine_id", "shift_id", "material_id"]).apply(
    lambda g: g["defective_units"].sum() / g["produced_units"].sum() * 100 if g["produced_units"].sum() > 1000 else None,
    include_groups=False
).dropna().sort_values(ascending=False)

print(f"Plant defect rate: {plant_defect_rate:.2f}%")
print(f"\nHighest-risk machine: {worst_machine.index[0]} ({worst_machine.iloc[0]:.2f}%)")
print(f"Highest-risk shift: {worst_shift.index[0]} ({worst_shift.iloc[0]:.2f}%)")
print(f"Highest-risk material: {worst_material.index[0]} ({worst_material.iloc[0]:.2f}%)")
print(f"\nHighest-risk combination: {worst_combo.index[0]} ({worst_combo.iloc[0]:.2f}%)")

Plant defect rate: 5.11%

Highest-risk machine: M07 (7.92%)
Highest-risk shift: Night (5.99%)
Highest-risk material: MAT09 (6.33%)

Highest-risk combination: ('M07', 'Night', 'MAT09') (11.27%)


In [4]:
importances = pd.DataFrame({
    "feature": feature_columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

sensor_groups = {
    "Tool Wear": ["tool_wear", "tool_wear_anomaly"],
    "Vibration": ["vibration", "vibration_deviation", "vibration_anomaly"],
    "Temperature": ["temperature", "temperature_deviation", "temp_anomaly"],
}

sensor_summary = {}
for group_name, cols in sensor_groups.items():
    group_score = importances[importances["feature"].isin(cols)]["importance"].sum()
    sensor_summary[group_name] = group_score

sensor_summary = dict(sorted(sensor_summary.items(), key=lambda x: x[1], reverse=True))
total = sum(sensor_summary.values())

print("Top associated sensor factors (grouped, relative share among sensor factors):")
rank = 1
for factor in sensor_summary:
    score = sensor_summary[factor]
    pct = round(score / total * 100, 1)
    print(str(rank) + ". " + factor + ": " + str(pct) + "%")
    rank = rank + 1

Top associated sensor factors (grouped, relative share among sensor factors):
1. Tool Wear: 54.7%
2. Vibration: 26.8%
3. Temperature: 18.6%


In [7]:
print("=" * 55)
print("YIELDWATCH ROOT-CAUSE ANALYSIS REPORT")
print("=" * 55)

print("\nPlant defect rate:")
print(str(round(plant_defect_rate, 2)) + "%")

print("\nHighest-risk machine:")
print(worst_machine.index[0] + " (" + str(round(worst_machine.iloc[0], 2)) + "%)")

print("\nHighest-risk shift:")
print(worst_shift.index[0] + " (" + str(round(worst_shift.iloc[0], 2)) + "%)")

print("\nHighest-risk material:")
print(worst_material.index[0] + " (" + str(round(worst_material.iloc[0], 2)) + "%)")

print("\nHighest-risk combination:")
combo_name = worst_combo.index[0][0] + " + " + worst_combo.index[0][1] + " + " + worst_combo.index[0][2]
print(combo_name + " (" + str(round(worst_combo.iloc[0], 2)) + "%)")

print("\nTop associated sensor factors:")
rank = 1
for factor in sensor_summary:
    score = sensor_summary[factor]
    pct = round(score / total * 100, 1)
    print(str(rank) + ". " + factor + " (" + str(pct) + "%)")
    rank = rank + 1

note = "Note: 'Associated with' reflects statistical patterns in historical data, not proven causation."

YIELDWATCH ROOT-CAUSE ANALYSIS REPORT

Plant defect rate:
5.11%

Highest-risk machine:
M07 (7.92%)

Highest-risk shift:
Night (5.99%)

Highest-risk material:
MAT09 (6.33%)

Highest-risk combination:
M07 + Night + MAT09 (11.27%)

Top associated sensor factors:
1. Tool Wear (54.7%)
2. Vibration (26.8%)
3. Temperature (18.6%)


In [8]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../database/yieldwatch.db")

runs = pd.read_sql("SELECT * FROM production_runs", conn)
sensors = pd.read_sql("SELECT * FROM sensor_readings", conn)
machines = pd.read_sql("SELECT * FROM machines", conn)
defects = pd.read_sql("SELECT * FROM defect_events", conn)

# ---- Export 1: Main fact table (runs + sensors combined) ----
fact_table = runs.merge(sensors.drop(columns=["timestamp"]), on="run_id")
fact_table["timestamp"] = pd.to_datetime(fact_table["timestamp"])
fact_table.to_csv("../powerbi/fact_production_runs.csv", index=False)
print("fact_production_runs.csv:", fact_table.shape)

# ---- Export 2: Machines dimension table ----
machines.to_csv("../powerbi/dim_machines.csv", index=False)
print("dim_machines.csv:", machines.shape)

# ---- Export 3: Defect events ----
defects.to_csv("../powerbi/defect_events.csv", index=False)
print("defect_events.csv:", defects.shape)

conn.close()

fact_production_runs.csv: (60000, 22)
dim_machines.csv: (20, 5)
defect_events.csv: (59959, 5)


In [9]:
import pickle

with open("../models/defect_model.pkl", "rb") as f:
    rf = pickle.load(f)
with open("../models/feature_columns.pkl", "rb") as f:
    feature_columns = pickle.load(f)

# Feature importance export
importances = pd.DataFrame({
    "feature": feature_columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)
importances.to_csv("../powerbi/feature_importance.csv", index=False)
print("feature_importance.csv:", importances.shape)

# Model metrics export - a small manually-built summary table
# (using the exact numbers you already got from Phase 10)
model_metrics = pd.DataFrame({
    "model": ["Logistic Regression", "Random Forest"],
    "accuracy": [0.8278, 0.8325],
    "precision": [0.8234, 0.8216],
    "recall": [0.8882, 0.9017],
    "f1_score": [0.8545, 0.8598],
    "roc_auc": [0.9222, 0.9236]
})
model_metrics.to_csv("../powerbi/model_metrics.csv", index=False)
print("model_metrics.csv:", model_metrics.shape)

feature_importance.csv: (27, 2)
model_metrics.csv: (2, 6)


In [10]:
risk_combo = runs.groupby(["machine_id", "shift_id", "material_id"]).apply(
    lambda g: pd.Series({
        "total_production": g["produced_units"].sum(),
        "defect_pct": g["defective_units"].sum() / g["produced_units"].sum() * 100
    }), include_groups=False
).reset_index()

risk_combo = risk_combo[risk_combo["total_production"] > 1000]

plant_avg = runs["defective_units"].sum() / runs["produced_units"].sum() * 100
risk_combo["risk_flag"] = "Normal"
risk_combo.loc[risk_combo["defect_pct"] > plant_avg * 1.2, "risk_flag"] = "Elevated Risk"
risk_combo.loc[risk_combo["defect_pct"] > plant_avg * 1.5, "risk_flag"] = "High Risk"

risk_combo = risk_combo.sort_values("defect_pct", ascending=False)
risk_combo.to_csv("../powerbi/risk_combinations.csv", index=False)
print("risk_combinations.csv:", risk_combo.shape)

risk_combinations.csv: (600, 6)
